In [654]:
import heapq
import numpy as np
import pandas as pd
import plotly.express as px
import collections
import itertools
import tqdm
from copy import deepcopy
from functools import lru_cache

In [655]:
np.random.seed(0)

In [656]:
def balance_priors(priors, random=True):
    total = np.sum(priors)
    if total == 1:
        return priors
    indices = priors == 0
    remainder = 1 - total
    if random:
        p = np.random.rand(indices.sum())
        p = (p / p.sum()) * remainder
    else:
        p = remainder / indices.sum()
    priors[indices] = p
    return priors

In [657]:
def bayesian_update(priors):
    if np.sum(priors) == 0:
        return np.zeros_like(priors)
    return priors / np.sum(priors)

In [658]:
def manipulation_thresholds(thresholds, priors, c):
    if np.sum(priors) == 0:
        return thresholds.copy()
    manip_thresholds = np.maximum(0, thresholds - (bayesian_update(priors) / c))
    return manip_thresholds

In [ ]:
def merge_classifiers(thresholds, priors, manip_thresholds, c):
    if len(thresholds) <= 1:
        return thresholds, priors, manip_thresholds
    
    merged_thresholds = [thresholds[-1]]
    merged_priors = [priors[-1]]
    merged_manip_thresholds = [manip_thresholds[-1]]

    for i in range(len(thresholds)-2,-1,-1):
        if thresholds[i] < merged_manip_thresholds[-1]:
            merged_thresholds.append(thresholds[i])
            merged_priors.append(priors[i])
            merged_manip_thresholds.append(manip_thresholds[i])
        else:
            merged_priors[-1] += priors[i]
            merged_manip_thresholds[-1] = np.maximum(0, merged_thresholds[-1] - (merged_priors[-1] / (c * np.sum(merged_priors)))).item()
    
    return np.array(merged_thresholds)[::-1], np.array(merged_priors)[::-1], np.array(merged_manip_thresholds)[::-1]

In [660]:
def accuracy_loss(thresholds, priors, manip_thresholds, threshold_true):
    losses = []
    for i in range(len(thresholds)):
        loss = np.abs(manip_thresholds[i] - threshold_true)
        losses.append(loss)
    losses = np.array(losses)
    posteriors = bayesian_update(priors)
    return np.dot(losses, posteriors)

In [661]:
def evaluate_partition(partition, thresholds, priors, threshold_true, c, return_all=False):
    thresholds_p = thresholds[partition]
    priors_p = priors[partition]
    manip_thresholds_p = manipulation_thresholds(thresholds_p, priors_p, c)
    thresholds_p, priors_p, manip_thresholds_p = merge_classifiers(thresholds_p, priors_p, manip_thresholds_p, c)
    acc_loss_p = accuracy_loss(thresholds_p, priors_p, manip_thresholds_p, threshold_true)
    if return_all:
        return acc_loss_p, thresholds_p, priors_p, manip_thresholds_p
    return acc_loss_p

def evaluate_system(partitions, thresholds, priors, threshold_true, c):
    acc_loss = 0.
    for partition in partitions:
        acc_loss_p = evaluate_partition(partition, thresholds, priors, threshold_true, c)
        acc_loss += acc_loss_p * np.sum(priors[partition])
    return acc_loss

In [662]:
def set_partitions(collection):
    if len(collection) == 1:
        yield [collection]
        return

    first = collection[0]
    for smaller in set_partitions(collection[1:]):
        for i in range(len(smaller)):
            yield smaller[:i] + [[first] + smaller[i]] + smaller[i+1:]
        yield [[first]] + smaller

In [663]:
def display_queue(Q, P):
    res = "[  "
    for (a_id, b_id) in Q:
        res += f"({P[a_id]}, {P[b_id]})  "
    res += "]"
    print(res)


def find_partitions_greedy_lex(thresholds, priors, threshold_true, c, display=False):
    P = {}
    next_id = 0
    partitions = [[i] for i in range(len(priors))]
    for block in partitions:
        P[next_id] = list(block)
        next_id += 1

    Q = collections.deque(itertools.combinations(P.keys(), 2))
    while Q:
        if display:
            display_queue(Q, P)
        a_id, b_id = Q.popleft()
        if a_id not in P.keys() or b_id not in P.keys():
            continue

        a = P[a_id]
        b = P[b_id]

        acc_loss_a = evaluate_partition(a, thresholds, priors, threshold_true, c)
        acc_loss_b = evaluate_partition(b, thresholds, priors, threshold_true, c)
        lhs = acc_loss_a * np.sum(priors[a]) + acc_loss_b * np.sum(priors[b])

        ab = sorted(a + b)
        acc_loss_ab = evaluate_partition(ab, thresholds, priors, threshold_true, c)
        rhs = acc_loss_ab * np.sum(priors[ab])

        if lhs - rhs > -1e-6:
            del P[a_id]
            del P[b_id]

            Q = collections.deque(
                (x, y)
                for (x, y) in Q
                if x not in {a_id, b_id} and y not in {a_id, b_id}
            )

            new_id = next_id
            next_id += 1
            P[new_id] = ab

            for p_id in P.keys():
                if p_id != new_id:
                    Q.append((new_id, p_id))

            # Q = collections.deque(sorted(Q, key=lambda x: P[x[0]]))
            
    return list(P.values())

In [664]:
def find_partitions_optimal(thresholds, priors, threshold_true, c):
    indices = list(range(len(thresholds)))
    partitions_set = list(set_partitions(indices))

    @lru_cache(maxsize=None)
    def evaluate_partition_cached(partition_tuple):
        partition = list(partition_tuple)
        acc_loss_p = evaluate_partition(
            partition, thresholds, priors, threshold_true, c
        )
        return acc_loss_p * np.sum(priors[partition])

    best_partition = None
    best_loss = np.inf

    for partitions in partitions_set:
        acc_loss = 0.0
        for partition in partitions:
            partition_tuple = tuple(sorted(partition))
            acc_loss += evaluate_partition_cached(partition_tuple)

        if acc_loss < best_loss:
            best_loss = acc_loss
            best_partition = deepcopy(partitions)

    return best_partition

In [665]:
runs = 100
N = np.arange(2, 11)
# C = np.arange(0., 10, 0.5) + 0.5
C = [0.5]

results = {"n": [], "run": [], "thresholds": [], "priors": [], "threshold_true": [], "c": [], "opt": [], "greedy": [], "acc_loss_opt": [], "acc_loss_greedy": [], "ratio": []}

for n in N:
    for run in tqdm.trange(runs, desc=f"[ n={n} ]"):
        thresholds = np.sort(np.random.rand(n)).round(6)
        priors = np.zeros_like(thresholds)
        balance_priors(priors, random=True)
        diff = 1 - np.sum(priors.round(6))
        priors = priors.round(6)
        priors[0] += diff
        # if abs(1-np.sum(priors)) > 1e-6:
        #     print(np.sum(priors))
        #     continue
        
        # threshold_true = np.random.rand()
        threshold_true = 0.9999999
        for c in C:
            partition_opt = find_partitions_optimal(thresholds, priors, threshold_true, c)
            partition_greedy = find_partitions_greedy_lex(thresholds, priors, threshold_true, c)

            acc_loss_opt = evaluate_system(partition_opt, thresholds, priors, threshold_true, c)
            acc_loss_greedy = evaluate_system(partition_greedy, thresholds, priors, threshold_true, c)

            ratio = (1 - acc_loss_opt)/(1- acc_loss_greedy)

            results["n"].append(n)
            results["run"].append(run)
            results["thresholds"].append(deepcopy(thresholds))
            results["priors"].append(deepcopy(priors))
            results["threshold_true"].append(threshold_true)
            results["c"].append(c)
            results["opt"].append(deepcopy(partition_opt))
            results["greedy"].append(deepcopy(partition_greedy))
            results["acc_loss_opt"].append(acc_loss_opt.item())
            results["acc_loss_greedy"].append(acc_loss_greedy.item())
            results["ratio"].append(ratio.item())

[ n=3 ]:   0%|          | 0/100 [00:00<?, ?it/s]


ValueError: can only convert an array of size 1 to a Python scalar

In [ ]:
df = pd.DataFrame(results)
print(df.shape)
df.head()

(900, 11)


,n,run,thresholds,priors,threshold_true,c,opt,greedy,acc_loss_opt,acc_loss_greedy,ratio
0,2,0,"[0.548814, 0.715189]","[0.525217, 0.474783]",1.0,0.5,"[[0, 1]]","[[0, 1]]",1.000000,1.000000,1.0
1,2,1,"[0.423655, 0.645894]","[0.329171, 0.670829]",1.0,0.5,"[[0], [1]]","[[0, 1]]",1.000000,1.000000,1.0
2,2,2,"[0.383442, 0.963663]","[0.59951, 0.40049]",1.0,0.5,"[[0, 1]]","[[0, 1]]",0.837317,0.837317,1.0
3,2,3,"[0.568045, 0.925597]","[0.449125, 0.550875]",1.0,0.5,"[[0], [1]]","[[0, 1]]",1.000000,1.000000,1.0
4,2,4,"[0.020218, 0.83262]","[0.472134, 0.527866]",1.0,0.5,"[[0], [1]]","[[0, 1]]",1.000000,1.000000,1.0


In [ ]:
df_im = df.copy()
df_im["n"] = df_im["n"].astype(str)
df_im["c"] = df_im["c"].astype(str)
px.scatter(df_im, x="n", y="ratio", color="c", hover_data=["run"])

In [ ]:
px.scatter(df_im, x="c", y="ratio", color="n", hover_data=["run"])

In [ ]:
i_max = df["ratio"].argmax()
df_max = df.iloc[[i_max]]
df_max

,n,run,thresholds,priors,threshold_true,c,opt,greedy,acc_loss_opt,acc_loss_greedy,ratio
285,4,85,"[0.032776, 0.672627, 0.82848, 0.852689]","[0.155024, 0.215304, 0.119833, 0.509839]",1.0,0.5,"[[0, 1, 2], [3]]","[[0, 1, 2, 3]]",0.886212,1.0,1.137879e+06


In [ ]:
px.scatter(df, x="acc_loss_opt", y="ratio", color="acc_loss_greedy")#.update_traces(marker=dict(size=3))

In [ ]:
px.scatter(df, x="threshold_true", y="acc_loss_opt")

In [ ]:
px.line(df[(df["n"] == 6) & (df["run"]==29)], x="c", y=["acc_loss_opt", "acc_loss_greedy"], markers=True)

In [ ]:
thresholds_ce = df_max["thresholds"].item()
priors_ce = df_max["priors"].item()

threshold_true_ce = df_max["threshold_true"].item()
c_ce = df_max["c"].item()


partition_opt = find_partitions_optimal(thresholds_ce, priors_ce, threshold_true_ce, c_ce)
partition_greedy = find_partitions_greedy_lex(thresholds_ce, priors_ce, threshold_true_ce, c_ce, True)

acc_loss_opt = evaluate_system(partition_opt, thresholds_ce, priors_ce, threshold_true_ce, c_ce)
acc_loss_greedy = evaluate_system(partition_greedy, thresholds_ce, priors_ce, threshold_true_ce, c_ce)
# partition_opt = df_max["opt"].item()
# partition_greedy = df_max["greedy"].item()

# acc_loss_opt = df_max["acc_loss_opt"].item()
# acc_loss_greedy = df_max["acc_loss_greedy"].item()


[  ([0], [1])  ([0], [2])  ([0], [3])  ([1], [2])  ([1], [3])  ([2], [3])  ]
[  ([2], [3])  ([0, 1], [2])  ([0, 1], [3])  ]
[  ([2, 3], [0, 1])  ]


In [ ]:
print(f"c         : {c_ce:.4f}")
print(f"threshold true: {threshold_true_ce}")
display(pd.DataFrame({"Thresholds": thresholds_ce, "Priors": priors_ce}).round(6).T)

print("Greedy")
print("------")
print(f"Partition: {sorted(partition_greedy)}")
print(f"Acc Loss : {acc_loss_greedy:.7f}")
print()
print("Optimal")
print("-------")
print(f"Partition: {sorted(partition_opt)}")
print(f"Acc Loss : {acc_loss_opt:.7f}")
print()
print(f"Ratio: {(1 - acc_loss_opt) / (1 - acc_loss_greedy):.4f}")

c         : 0.5000
threshold true: 0.9999999


,0,1,2,3
Thresholds,0.032776,0.672627,0.828480,0.852689
Priors,0.155024,0.215304,0.119833,0.509839


Greedy
------
Partition: [[0, 1, 2, 3]]
Acc Loss : 0.9999999

Optimal
-------
Partition: [[0, 1, 2], [3]]
Acc Loss : 0.8862121

Ratio: 1137879.4928


In [ ]:
a, b = [0,1,2], [3]

acc_loss_a, thresholds_a, priors_a, manip_thresholds_a = evaluate_partition(a, thresholds_ce, priors_ce, threshold_true_ce, c_ce, True)
acc_loss_b, thresholds_b, priors_b, manip_thresholds_b = evaluate_partition(b, thresholds_ce, priors_ce, threshold_true_ce, c_ce, True)

lhs = acc_loss_a * np.sum(priors_ce[a]) + acc_loss_b * np.sum(priors_ce[b])
ab = sorted(a + b)

acc_loss_ab, thresholds_ab, priors_ab, manip_thresholds_ab = evaluate_partition(ab, thresholds_ce, priors_ce, threshold_true_ce, c_ce, True)
rhs = acc_loss_ab * np.sum(priors_ce[ab])

print(f"    threshold true: {threshold_true_ce}")
print(f"                 c: {c_ce}")
print(f"                 a: {a}")
print(f"                 b: {b}")
print(f"   accuracy loss a: {acc_loss_a:.7f}")
print(f"   accuracy loss b: {acc_loss_b:.7f}")
print(f"  accuracy loss ab: {acc_loss_ab:.7f}")
print(f"               LHS: {lhs:.7f}")
print(f"               RHS: {rhs:.7f}")
print(f"            merge?: {lhs - rhs > -1e-9}")
print()
print(f"      thresholds a: {thresholds_a}")
print(f"          priors a: {priors_a}")
print(f"      thresholds b: {thresholds_b}")
print(f"          priors b: {priors_b}")
print(f"     thresholds ab: {thresholds_ab}")
print(f"         priors ab: {priors_ab}")
print(f" manip threshold a: {manip_thresholds_a}")
print(f" manip threshold b: {manip_thresholds_b}")
print(f"manip threshold ab: {manip_thresholds_ab}")

    threshold true: 0.9999999
                 c: 0.5
                 a: [0, 1, 2]
                 b: [3]
   accuracy loss a: 0.9999999
   accuracy loss b: 0.9999999
  accuracy loss ab: 0.9999999
               LHS: 0.9999999
               RHS: 0.9999999
            merge?: True

      thresholds a: [0.82848]
          priors a: [0.490161]
      thresholds b: [0.852689]
          priors b: [0.509839]
     thresholds ab: [0.852689]
         priors ab: [1.]
 manip threshold a: [0.]
 manip threshold b: [0.]
manip threshold ab: [0.]


In [ ]:
acc_loss_a

array([0.9999999])